# 02a - Fetch GEE Observations

## Purpose
Fetch or restore raw Google Earth Engine observations for each sample/date. This is the slow network-bound stage.

## Inputs
- `data/processed/sample_index.csv`
- Per-sample frame dates from `frame_metadata.csv`
- Google Earth Engine project ID

## Outputs
- `data/raw/<label>/<sample_id>/gee_observations.csv`
- `data/raw/<label>/<sample_id>/gee_feature_metadata.json`

## Notes
This notebook must not create features, targets, rolling windows, imputation outputs, or combined training files. Set `RUN_GEE_FETCH = True` only when you intentionally want to call Earth Engine.

## 1. Configure GEE Fetch Settings
Defines project paths, the ignored Earth Engine credential path, fetch limits, cache behavior, and the explicit `RUN_GEE_FETCH` safety switch.

In [ ]:
from pathlib import Path
import sys
import importlib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_INDEX_PATH = DATA_DIR / "processed" / "sample_index.csv"
GEE_CREDENTIALS_PATH = PROJECT_ROOT / "config" / "gee_credentials.json"
SAMPLE_LIMIT = None
FORCE_REFETCH = False
RUN_GEE_FETCH = True

## 2. Load GEE Fetch Helpers
Reloads the GEE helper module and binds only the raw-observation fetch/cache functions used by this notebook.

In [ ]:
import src.features.gee_features as gee_feature_module
importlib.reload(gee_feature_module)

FeatureExtractionConfig = gee_feature_module.FeatureExtractionConfig
load_gee_project_id = gee_feature_module.load_gee_project_id
extract_raw_observations_from_gee = gee_feature_module.extract_raw_observations_from_gee

## 3. Build Fetch Config
Creates the fetch configuration and optionally copies older processed observation caches into the new `data/raw` layout.

In [ ]:
GEE_PROJECT_ID = load_gee_project_id(GEE_CREDENTIALS_PATH, project_root=PROJECT_ROOT)
print("Loaded GEE project id from environment or local credentials file")

config = FeatureExtractionConfig(
    gee_project_id=GEE_PROJECT_ID,
    gee_credentials_path=str(GEE_CREDENTIALS_PATH),
    data_dir=str(DATA_DIR),
    sample_index_csv=str(SAMPLE_INDEX_PATH),
    force=FORCE_REFETCH,
    verbose=True,
    log_feature_groups=True,
)


## 4. Check Raw Observation Progress
Shows a simple progress bar across all samples and reports how many raw `gee_observations.csv` files are already available before any optional Earth Engine fetch runs.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

sample_index_for_progress = pd.read_csv(SAMPLE_INDEX_PATH)
if SAMPLE_LIMIT is not None:
    sample_index_for_progress = sample_index_for_progress.head(SAMPLE_LIMIT)

ready_paths = []
missing_paths = []
progress = tqdm(
    sample_index_for_progress.itertuples(index=False),
    total=len(sample_index_for_progress),
    desc="Raw GEE observation files",
    unit="sample",
)

for sample in progress:
    observation_path = Path(sample.gee_observations_csv)
    if observation_path.exists():
        ready_paths.append(observation_path)
    else:
        missing_paths.append(observation_path)
    progress.set_postfix(ready=len(ready_paths), missing=len(missing_paths))

print(f"Ready raw observation files: {len(ready_paths)}/{len(sample_index_for_progress)}")
if missing_paths:
    print("Missing raw observation files:")
    for path in missing_paths[:10]:
        print(f"- {path}")
    if len(missing_paths) > 10:
        print(f"... and {len(missing_paths) - 10} more")

## 5. Optional Earth Engine Fetch
Calls Google Earth Engine only if `RUN_GEE_FETCH` is set to `True`. During the actual fetch, the helper displays `tqdm` progress bars for sample and frame-level progress.

In [ ]:
if RUN_GEE_FETCH:
    written_paths = extract_raw_observations_from_gee(config, sample_limit=SAMPLE_LIMIT)
    print(f"Raw observation files ready: {len(written_paths)}")
else:
    print("RUN_GEE_FETCH is False. Set it to True only when you want to call Google Earth Engine.")